# 仿射密码 (Affine Cipher) — 自学课程

**分类**：古典密码学

**内容简介**：单表替换密码，基于线性变换加密，是凯撒密码的推广，依赖模运算与乘法逆元



## 学习目标

- 理解算法规则与数学表达
- 实现加解密函数，并写基本测试
- 通过小实验观察安全性与攻击方法
- 能解释常见踩坑并独立调试



## 前置知识与符号约定

- 字母表：仅使用大写 A-Z
- 映射：$A\rightarrow 0,\dots,Z\rightarrow 25$
- 模运算：$x\bmod 26$ 结果落在 $\{0,\dots,25\}$

> 如果你希望支持中文/标点，本课程也会在后续练习中给出扩展思路。



In [ ]:
# 课程通用工具：字母映射与规范化
import string
from collections import Counter, defaultdict
import math, random

ALPHABET = string.ascii_uppercase
A2I = {ch:i for i,ch in enumerate(ALPHABET)}
I2A = {i:ch for i,ch in enumerate(ALPHABET)}

def normalize(text: str) -> str:
    """仅保留 A-Z 并转大写"""
    return ''.join(ch for ch in text.upper() if ch in ALPHABET)

def keep_nonletters_encrypt(text: str, enc_func):
    """对字母加密，非字母原样保留（用于扩展练习）"""
    out=[]
    for ch in text:
        if ch.upper() in ALPHABET:
            out.append(enc_func(ch.upper()))
        else:
            out.append(ch)
    return ''.join(out)

print(normalize("Hello, World! 123"))
# 预期输出：HELLOWORLD



# Step 1：把算法写成数学模型

我们用统一抽象描述密码：

- 加密：$E_k: \mathcal{P}\to\mathcal{C}$
- 解密：$D_k: \mathcal{C}\to\mathcal{P}$
- 正确性：

$$D_k(E_k(p))=p$$

这能帮助你：
- 写出可测试的实现
- 分析密钥空间大小
- 讨论攻击模型（已知明文、选择明文等）



## 自检小测

1) 什么是“模 26”运算？为什么它会让结果回到 0..25？
2) 为什么实现中必须统一 A→0 的映射？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



> 常见错误 / 踩坑点 / 调试提示：
>
> - 不要混用 A→1 与 A→0 两种映射。
> - 记得对 k 做 k%=26；否则 k>26 时结果可能不符合预期。
> - 先写 round-trip 测试：decrypt(encrypt(p,k),k)==p。



# Step 2：仿射密码公式与可逆条件

仿射密码：

$$E_{a,b}(x)=(a x + b)\bmod 26$$

解密需要 $a^{-1}$：

$$D_{a,b}(y)=a^{-1}(y-b)\bmod 26$$

可逆条件：

$$\gcd(a,26)=1$$



In [ ]:
# Step 2：扩展欧几里得与模逆（本课程自带）
def egcd(a: int, b: int):
    if b == 0:
        return a, 1, 0
    g, x1, y1 = egcd(b, a % b)
    return g, y1, x1 - (a // b) * y1

def inv_mod(a: int, n: int):
    g, x, _ = egcd(a, n)
    if g != 1:
        return None
    return x % n

print(inv_mod(5,26), inv_mod(2,26))
# 预期输出：一个数字 + None



In [ ]:
# Step 3：加解密实现
def affine_encrypt(pt: str, a: int, b: int) -> str:
    if inv_mod(a,26) is None:
        raise ValueError("a 不可逆")
    t=normalize(pt)
    return ''.join(I2A[(a*A2I[ch]+b)%26] for ch in t)

def affine_decrypt(ct: str, a: int, b: int) -> str:
    ainv=inv_mod(a,26)
    if ainv is None:
        raise ValueError("a 不可逆")
    t=normalize(ct)
    return ''.join(I2A[(ainv*(A2I[ch]-b))%26] for ch in t)

ct=affine_encrypt("AFFINECIPHER",5,8)
print(ct)
print(affine_decrypt(ct,5,8))
# 预期输出：
# IHHWVCSWFRCP
# AFFINECIPHER



# Step 4：密钥空间大小估算

合法的 $a$ 必须与 26 互素。可用的 $a$ 数量是欧拉函数：

$$\varphi(26)=\varphi(2\cdot 13)=(2-1)(13-1)=12$$

$b$ 有 26 种，所以密钥空间大小约为：

$$|\mathcal{K}|=12\times 26=312$$

> 312 仍然很小，仍可暴力破解。



In [ ]:
# Step 4：列出所有合法 a
valid_a=[a for a in range(26) if inv_mod(a,26) is not None]
print(len(valid_a), valid_a)
# 预期输出：12 和一个列表



# Step 5：暴力破解

遍历所有 $(a,b)$，用评分器选最可能的明文。



In [ ]:
# Step 5：仿射爆破
def affine_crack(ct: str, topk=5):
    ct=normalize(ct)
    cand=[]
    for a in valid_a:
        for b in range(26):
            pt=affine_decrypt(ct,a,b)
            cand.append((score_english_like(pt), a, b, pt))
    cand.sort(reverse=True)
    return cand[:topk]

ct=affine_encrypt("THISISATESTMESSAGE",5,8)
for s,a,b,pt in affine_crack(ct, topk=5):
    print(s,a,b,pt)
# 预期输出：应出现 a=5,b=8 的候选



# Step 6：已知明文攻击（两对样本）

若知道两对明文密文 $(x_1,y_1),(x_2,y_2)$：

$$y_1-y_2 \equiv a(x_1-x_2)\pmod{26}$$

若 $(x_1-x_2)$ 可逆，就能求出 $a$，再求 $b$。



In [ ]:
# Step 6：两对样本恢复 (a,b)
def recover_affine(x1, y1, x2, y2, n=26):
    dx=(x1-x2)%n
    dy=(y1-y2)%n
    inv_dx=inv_mod(dx,n)
    if inv_dx is None:
        return None
    a=(dy*inv_dx)%n
    b=(y1-a*x1)%n
    return a,b

a0,b0=5,8
pt="AF"
ct=affine_encrypt(pt,a0,b0)
x1,x2=A2I["A"],A2I["F"]
y1,y2=A2I[ct[0]],A2I[ct[1]]
print(ct, recover_affine(x1,y1,x2,y2))
# 预期输出：恢复 (5,8)



## 练习

1) 把字母表扩展为 27 个符号（含空格），重新计算可逆条件与密钥空间。
2) 实现一个更强评分器并比较 topk 命中率。
3) 思考：仿射密码为什么仍属于“线性变换”？线性结构为何危险？


## 总结与延伸

- 你已经完成：规则→数学→实现→测试→攻击/评估。
- 下一步可以：
  - 为实现添加更多字符集与格式化规则
  - 写更强的评分器做自动破译
  - 把多个古典密码组合，体验“组合不等于安全”

> 重要：古典密码主要用于学习思想；真实系统请使用经过标准化与审计的现代密码算法。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。

